# Route B Pipeline 1: Histogram Gradient Boosting

Predicts `career_games_started` from combine measurements, validates locally, and saves held-out predictions (sections 1–5).
Sections 6–8 add a second prediction target, **rookie-season scrimmage snaps**, which gives every draft class the same one-season exposure window.

Each target is compared across nested feature sets (combine → + NGS scores → + draft pick) and two models (Ridge, shallow gradient boosting) on the same folds.
All outputs are written to `outputs/`.

Reproduce from the repo root: `uv sync && uv run jupyter nbconvert --to notebook --execute --inplace kaggle_pipeline.ipynb`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Config (edit these)

In [2]:
DATA_DIR = Path("data")
COMBINE_PATH = DATA_DIR / "combine_results.csv"
CAREER_PATH = DATA_DIR / "player_career_successes.csv"
PLAYERS_PATH = DATA_DIR / "players.csv"
GAMES_PATH = DATA_DIR / "games.csv"
PLAYER_PLAY_PATH = DATA_DIR / "player_play.csv"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = "career_games_started"
RANDOM_STATE = 307
CV_REPEATS = 3          # repeated 5-fold CV: 5 folds x 3 repeats = 15 scores per model
UNDRAFTED_PICK = 300    # undrafted players are placed after the last draft pick (plus an `undrafted` flag)

# Career totals depend on how many seasons a player has had (2023 class: 3, 2025 class: 1),
# so draft_year is kept as an exposure control for TARGET. It is dropped for the rookie-season target.
EXPOSURE = ["draft_year"]
COMBINE = ["combine_position", "combine_height", "combine_weight", "hand_size", "arm_length",
           "wing_span", "forty", "bench_reps", "vertical", "broad_jump", "three_cone",
           "short_shuttle", "ten_yd_split"]
NGS = ["ngs_athleticism_score", "ngs_college_production_score", "ngs_final_score"]
DRAFT = ["log_draft_pick", "undrafted"]

# Nested feature sets, all evaluated on the same folds. "Draft pick only" is a reference:
# it shows how much the NFL's own evaluation already explains.
FEATURE_SETS = {
    "Combine": EXPOSURE + COMBINE,
    "Combine + NGS": EXPOSURE + COMBINE + NGS,
    "Combine + NGS + draft pick": EXPOSURE + COMBINE + NGS + DRAFT,
    "Draft pick only": EXPOSURE + ["combine_position"] + DRAFT,
}
MAIN_SET = "Combine + NGS"   # pre-draft information only; used for the saved holdout predictions
FEATURES = FEATURE_SETS[MAIN_SET]

## 2. Preparation: one row per player (combine + career outcome + draft position)

In [3]:
combine = pd.read_csv(COMBINE_PATH)
career = pd.read_csv(CAREER_PATH)
players = pd.read_csv(PLAYERS_PATH)

players["undrafted"] = players["draft_overall_pick"].isna().astype(int)
players["log_draft_pick"] = np.log(players["draft_overall_pick"].fillna(UNDRAFTED_PICK))

all_features = sorted({c for cols in FEATURE_SETS.values() for c in cols} - set(DRAFT))
missing_cols = [c for c in all_features if c not in combine.columns]
if missing_cols:
    raise KeyError(f"Feature columns not found in combine data: {missing_cols}")

data = (combine
        .merge(career[["nfl_id", TARGET]], on="nfl_id", how="inner", validate="one_to_one")
        .merge(players[["nfl_id"] + DRAFT], on="nfl_id", how="left", validate="one_to_one"))
assert data[DRAFT].notna().all().all(), "every combine player should appear in players.csv"
data[TARGET] = pd.to_numeric(data[TARGET], errors="coerce")
data = data.dropna(subset=[TARGET]).reset_index(drop=True)
train = data  # name kept from the original pipeline
X, y = data[FEATURES], data[TARGET]

numeric = X.select_dtypes(include="number").columns.tolist()
categorical = X.select_dtypes(exclude="number").columns.tolist()
print("Training table:", data.shape)
print("Numeric:", numeric)
print("Categorical:", categorical)
print("Undrafted players:", int(data["undrafted"].sum()))
print("\nMean", TARGET, "by draft class (exposure differs by class):")
print(data.groupby("draft_year")[TARGET].agg(["count", "mean", "median"]).round(2))

Training table: (510, 21)
Numeric: ['draft_year', 'combine_height', 'combine_weight', 'hand_size', 'arm_length', 'wing_span', 'forty', 'bench_reps', 'vertical', 'broad_jump', 'three_cone', 'short_shuttle', 'ten_yd_split', 'ngs_athleticism_score', 'ngs_college_production_score', 'ngs_final_score']
Categorical: ['combine_position']
Undrafted players: 127

Mean career_games_started by draft class (exposure differs by class):
            count   mean  median
draft_year                      
2023          160  12.48     5.0
2024          177   8.00     2.0
2025          173   3.43     0.0


## 3. Pipeline definition (preprocessing lives inside the pipeline)

Missing-value indicators are added for numeric columns, because skipping a drill (e.g. bench or three-cone) can itself be informative.
Ridge gets standardized inputs and a cross-validated penalty. Gradient boosting is kept shallow (`max_depth=3`, `min_samples_leaf=20`) because there are only about 380 training players.

In [4]:
def make_pipeline(cols=None, kind="hgb"):
    cols = FEATURES if cols is None else cols
    num = data[cols].select_dtypes(include="number").columns.tolist()
    cat = [c for c in cols if c not in num]

    num_steps = [("imp", SimpleImputer(strategy="median", add_indicator=True))]
    if kind == "ridge":
        num_steps.append(("scale", StandardScaler()))
    transformers = [("num", Pipeline(num_steps), num)]
    if cat:
        transformers.append(("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat))

    if kind == "ridge":
        model = RidgeCV(alphas=np.logspace(-2, 3, 30))
    else:
        model = HistGradientBoostingRegressor(
            max_iter=200, learning_rate=0.05, max_depth=3, min_samples_leaf=20,
            l2_regularization=1.0, random_state=RANDOM_STATE)
    return Pipeline([("pre", ColumnTransformer(transformers)), ("model", model)])

## 4. Local validation: 75/25 holdout and repeated 5-fold CV

Both the holdout split and the CV folds are stratified by position and shared by every model and feature set (and by the rookie-snaps target in section 7), so comparisons are paired.
The mean baseline is scored on the same CV folds. `CV_MAE_sd` is the spread across the 15 fold scores.

In [5]:
positions = data["combine_position"]
idx_tr, idx_val = train_test_split(data.index, test_size=0.25, random_state=RANDOM_STATE,
                                   stratify=positions)
cv_folds = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=CV_REPEATS,
                                        random_state=RANDOM_STATE).split(data, positions))
X_tr, X_val, y_tr, y_val = X.loc[idx_tr], X.loc[idx_val], y.loc[idx_tr], y.loc[idx_val]
MODELS = {"Ridge": "ridge", "HistGradientBoosting": "hgb"}


def evaluate(target, feature_sets):
    """Score a mean baseline plus every (feature set, model) pair on the shared holdout and CV folds."""
    first_cols = next(iter(feature_sets.values()))
    candidates = [("Mean baseline", "-", DummyRegressor(strategy="mean"), first_cols)]
    candidates += [(m, s, make_pipeline(cols, kind), cols)
                   for s, cols in feature_sets.items() for m, kind in MODELS.items()]
    rows, fitted = [], {}
    for model_name, set_name, model, cols in candidates:
        Xc = data[cols]
        cv = -cross_val_score(clone(model), Xc, target, cv=cv_folds,
                              scoring="neg_mean_absolute_error")
        model.fit(Xc.loc[idx_tr], target.loc[idx_tr])
        p = model.predict(Xc.loc[idx_val])
        rows.append({"Model": model_name, "Features": set_name,
                     "MAE": mean_absolute_error(target.loc[idx_val], p),
                     "RMSE": np.sqrt(mean_squared_error(target.loc[idx_val], p)),
                     "R2": r2_score(target.loc[idx_val], p),
                     "CV_MAE_mean": cv.mean(), "CV_MAE_sd": cv.std()})
        fitted[(model_name, set_name)] = model
    return pd.DataFrame(rows), fitted


val, fitted = evaluate(y, FEATURE_SETS)
base = fitted[("Mean baseline", "-")]
pipe = fitted[("HistGradientBoosting", MAIN_SET)]
val.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
val.round(3)

,Model,Features,MAE,RMSE,R2,CV_MAE_mean,CV_MAE_sd
0,Mean baseline,-,9.446,12.946,-0.012,8.838,0.379
1,Ridge,Combine,8.765,12.176,0.105,8.284,0.357
2,HistGradientBoosting,Combine,9.356,13.288,-0.066,8.532,0.580
3,Ridge,Combine + NGS,7.377,10.293,0.360,6.791,0.230
4,HistGradientBoosting,Combine + NGS,6.908,10.168,0.376,6.374,0.439
5,Ridge,Combine + NGS + draft pick,6.636,9.446,0.461,5.883,0.389
6,HistGradientBoosting,Combine + NGS + draft pick,5.368,8.236,0.591,5.058,0.363
7,Ridge,Draft pick only,6.580,9.344,0.473,5.867,0.375
8,HistGradientBoosting,Draft pick only,5.461,8.395,0.575,4.947,0.358


## 5. Inspect and save held-out predictions

In [6]:
holdout = pd.DataFrame({
    "nfl_id": data.loc[idx_val, "nfl_id"],
    "combine_position": data.loc[idx_val, "combine_position"],
    "draft_year": data.loc[idx_val, "draft_year"],
    "actual_games_started": y_val,
    "predicted_games_started": np.clip(pipe.predict(X_val), 0, None),  # starts cannot be negative
})
holdout["absolute_error"] = (holdout["actual_games_started"] - holdout["predicted_games_started"]).abs()
holdout.to_csv(OUTPUT_DIR / "holdout_predictions.csv", index=False)
print(f"Wrote {OUTPUT_DIR / 'holdout_predictions.csv'}", holdout.shape, f"(model: HistGradientBoosting, {MAIN_SET})")
holdout.sort_values("absolute_error", ascending=False).head(15)

Wrote outputs/holdout_predictions.csv (128, 6) (model: HistGradientBoosting, Combine + NGS)


,nfl_id,combine_position,draft_year,actual_games_started,predicted_games_started,absolute_error
77,55942,DL,2023,49,8.054700,40.945300
57,55947,DL,2023,41,5.345352,35.654648
60,55984,DB,2023,33,3.965515,29.034485
13,55914,DL,2023,40,12.987195,27.012805
158,55891,OL,2023,48,21.395831,26.604169
291,57198,OL,2024,31,4.816377,26.183623
85,55867,DL,2023,44,20.117057,23.882943
66,55912,DB,2023,37,14.207558,22.792442
50,56078,DL,2023,21,2.744241,18.255759
155,55924,OL,2023,50,32.326154,17.673846


## 6. Additional target: rookie-season scrimmage snaps

`career_games_started` mixes ability with time in the league. The rookie-season target counts the regular-season plays each player
appears on in `player_play.csv` during their draft year, so every class has exactly one season of exposure.
About a quarter of players have zero rookie snaps and the distribution is right-skewed, so models are fit on `log1p(rookie_snaps)`.

In [ ]:
games = pd.read_csv(GAMES_PATH, usecols=["game_id", "season", "season_type"])
plays = pd.read_csv(PLAYER_PLAY_PATH, usecols=["game_id", "nfl_id"])
plays = (plays.merge(games, on="game_id", how="inner", validate="many_to_one")
              .merge(data[["nfl_id", "draft_year"]], on="nfl_id", how="inner"))
rookie_plays = plays[(plays["season"] == plays["draft_year"]) & (plays["season_type"] == "REG")]

data["rookie_snaps"] = data["nfl_id"].map(rookie_plays.groupby("nfl_id").size()).fillna(0).astype(int)
data["rookie_games"] = data["nfl_id"].map(rookie_plays.groupby("nfl_id")["game_id"].nunique()).fillna(0).astype(int)
y_rookie = np.log1p(data["rookie_snaps"])

# For the 2025 class the rookie season is the whole career, so this should track career scrimmage snaps.
c25 = data["draft_year"] == 2025
scrim = career.set_index("nfl_id").loc[data.loc[c25, "nfl_id"], ["career_offensive_snaps", "career_defensive_snaps"]].sum(axis=1)
print("Corr(rookie_snaps, career scrimmage snaps), 2025 class:",
      round(np.corrcoef(data.loc[c25, "rookie_snaps"], scrim.to_numpy())[0, 1], 3))
print("Share with zero rookie snaps:", round((data["rookie_snaps"] == 0).mean(), 3))
print("\nRookie snaps by draft class (exposure is now comparable):")
print(data.groupby("draft_year")["rookie_snaps"].agg(["count", "mean", "median"]).round(1))

Corr(rookie_snaps, career scrimmage snaps), 2025 class: 0.964
Share with zero rookie snaps: 0.259

Rookie snaps by draft class (exposure is now comparable):
            count   mean  median
draft_year                      
2023          160  242.4    96.0
2024          177  248.1   113.0
2025          173  230.3    96.0


## 7. Local validation for rookie snaps (same holdout, same CV folds)

`draft_year` is dropped from every feature set because exposure is already equal. Metrics are on the log scale.

In [8]:
ROOKIE_FEATURE_SETS = {name: [c for c in cols if c not in EXPOSURE] for name, cols in FEATURE_SETS.items()}
rookie_val, rookie_fitted = evaluate(y_rookie, ROOKIE_FEATURE_SETS)
rookie_val.to_csv(OUTPUT_DIR / "rookie_validation_metrics.csv", index=False)
rookie_val.round(3)

,Model,Features,MAE,RMSE,R2,CV_MAE_mean,CV_MAE_sd
0,Mean baseline,-,2.297,2.569,-0.007,2.246,0.106
1,Ridge,Combine,2.244,2.568,-0.006,2.234,0.100
2,HistGradientBoosting,Combine,2.281,2.733,-0.140,2.210,0.148
3,Ridge,Combine + NGS,1.836,2.164,0.286,1.867,0.093
4,HistGradientBoosting,Combine + NGS,1.818,2.194,0.265,1.803,0.075
5,Ridge,Combine + NGS + draft pick,1.500,1.833,0.487,1.538,0.081
6,HistGradientBoosting,Combine + NGS + draft pick,1.431,1.791,0.510,1.451,0.076
7,Ridge,Draft pick only,1.453,1.781,0.516,1.505,0.080
8,HistGradientBoosting,Draft pick only,1.276,1.642,0.589,1.404,0.078


## 8. Inspect and save rookie-snaps held-out predictions (back-transformed to snaps)

In [9]:
rookie_pipe = rookie_fitted[("HistGradientBoosting", MAIN_SET)]
rookie_base = rookie_fitted[("Mean baseline", "-")]
Xr_val = data.loc[idx_val, ROOKIE_FEATURE_SETS[MAIN_SET]]

rookie_holdout = pd.DataFrame({
    "nfl_id": data.loc[idx_val, "nfl_id"],
    "combine_position": data.loc[idx_val, "combine_position"],
    "draft_year": data.loc[idx_val, "draft_year"],
    "actual_rookie_snaps": data.loc[idx_val, "rookie_snaps"],
    "predicted_rookie_snaps": np.clip(np.expm1(rookie_pipe.predict(Xr_val)), 0, None),
})
rookie_holdout["absolute_error"] = (rookie_holdout["actual_rookie_snaps"] - rookie_holdout["predicted_rookie_snaps"]).abs()
rookie_holdout.to_csv(OUTPUT_DIR / "rookie_holdout_predictions.csv", index=False)

base_snaps = np.expm1(rookie_base.predict(Xr_val))
print(f"Holdout MAE in snaps -- mean-of-log baseline: "
      f"{mean_absolute_error(rookie_holdout['actual_rookie_snaps'], base_snaps):.1f}, "
      f"HistGradientBoosting ({MAIN_SET}): {rookie_holdout['absolute_error'].mean():.1f}")
print(f"Wrote {OUTPUT_DIR / 'rookie_holdout_predictions.csv'}", rookie_holdout.shape)
rookie_holdout.sort_values("absolute_error", ascending=False).head(15)

Holdout MAE in snaps -- mean-of-log baseline: 245.8, HistGradientBoosting (Combine + NGS): 194.5
Wrote outputs/rookie_holdout_predictions.csv (128, 6)


,nfl_id,combine_position,draft_year,actual_rookie_snaps,predicted_rookie_snaps,absolute_error
291,57198,OL,2024,998,29.576851,968.423149
472,58262,OL,2025,1088,160.678637,927.321363
77,55942,DL,2023,968,59.671296,908.328704
492,58253,OL,2025,1082,331.164164,750.835836
128,55982,OL,2023,772,27.224368,744.775632
495,58242,OL,2025,982,253.551173,728.448827
155,55924,OL,2023,1163,446.910102,716.089898
500,58231,OL,2025,1054,354.477504,699.522496
65,55888,DB,2023,844,160.606449,683.393551
148,55922,OL,2023,755,96.183429,658.816571
